# 08 - Reading the dark bound

**Purpose.** To explain what session 03 found: what was captured, why the dark current comes out
as a bound rather than a value, what the discrete offset state is and how it was caught, and what
the project can and cannot do now that it could not do before. `07` is the notebook that *made*
these numbers, and is written for someone checking the work. This one is written for someone
deciding what to do next.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `dark_constants.json`, `pedestal_series.csv`, `dark_blocks.csv`, and the earlier
sessions' `bias_constants.json` and `ptc_constants.json`. Where arithmetic is done below it is
done on published numbers, to show what a published number is worth; if any of it disagreed with
`results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`.** Why noise is measured from a difference of two frames, why a
variance on a plane mean falls as 1/N, what a residual about a trend means - all of that is
explained there, on these same conventions, and is cited rather than re-derived.

**The headline.** The night was designed to bound `D` and it did: **`|D| < 5.34e-4 e-/px/s` at
-10 C**, against L14's inherited `< 1e-2`. That bound retires the dark term from the model. But
the reason it is a bound is not the reason the protocol expected, and that reason is the session's
real finding: **the camera's black level occupies discrete states about one ADC count apart**, and
it moves between them during a run at a fixed configuration.

Three things are worth your attention:

1. the state step is **larger than the entire dark excess this night measures at any exposure** -
   1.6 times the 600 s excess and 4.6 times the 300 s one - so dark-minus-bias measures the state
   whenever the two frames are not in the same one;
2. it is **rejectable at about a thousand sigma** on a plane mean, so the cost is a filter and not
   a lost night - but the filter is not optional in any future session;
3. the two results defined as differences **in space** rather than in time - DSNU and the glow
   gradient - are immune to it by construction, and they are the night's firm numbers.


In [ ]:
import json
import pathlib
import re
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"

ped = pd.read_csv(RESULTS / "pedestal_series.csv")
blocks = pd.read_csv(RESULTS / "dark_blocks.csv")
with open(RESULTS / "dark_constants.json") as fh:
    K = json.load(fh)
with open(RESULTS / "bias_constants.json") as fh:
    K1 = json.load(fh)
with open(RESULTS / "ptc_constants.json") as fh:
    K2 = json.load(fh)

GAIN, OFFSET = 250, K1["project_offset"]["value"]
G_E = K2["system_gain"]["value"][str(GAIN)]           # e- per ADC count at gain 250
G_E_ERR = K2["system_gain"]["uncertainty"][str(GAIN)]
R_COUNTS = 1.728                                       # session 01 at gain 250, offset 15
SKY_E_PER_S = 1.594                                    # L32, green, unfiltered, Bortle 5-6

STEP = K["offset_state_step"]["value"]
INCIDENCE = K["offset_state_incidence"]["value"]
D_BOUND = K["dark_current_bound"]["value"]
DSNU = K["dsnu_300s"]["value"]
GLOW = K["glow_gradient"]["value"]
ETA = {int(k): v for k, v in K["eta_comb"]["value"].items()}
N_FRAMES = K["offset_state_step"]["source_frames"]

PLANES = ["R", "G1", "G2", "B"]
roi = blocks[~blocks.full_frame]

print(f"{N_FRAMES} frames shot {K['dark_current_bound']['measured_on']} at gain {GAIN}, "
      f"offset {OFFSET}, {K['setpoint_held']['value'][0]} C")
print(f"{(~ped.full_frame).sum()} bias blocks at the ROI over "
      f"{ped[~ped.full_frame].t_min.max():.0f} min (+{ped.full_frame.sum()} at full frame), "
      f"{len(blocks)} dark blocks at {sorted(set(blocks.exptime))} s")
print(f"g at gain {GAIN} = {G_E:.5f} +/- {G_E_ERR:.5f} e- per ADC count (session 02)")


---

## 1. What the night looked like

Fifteen bias blocks at the session ROI and fourteen dark blocks, interleaved on a 20-minute clock
so that no dark is ever more than ten minutes from a pedestal measurement, plus two more bias
blocks at full frame bracketing the full-frame darks. That cadence is the whole design: the
protocol's rule 2 established, before the data existed, that the error bar on `D` would be
*pedestal wander* and not frame count - 11.7 counts across a bracket fakes `D = 1e-2 e-/px/s`,
and 0.12 counts fakes `1e-4`.

**One block is missing from the published tables**, and its absence is the finding announcing
itself: block 27's two 600 s darks were *both* in the far state, so it has no clean frames and no
row. Three 600 s blocks survive, not four. Section 3 says why.

The ladder runs 1 s, 60 s, then eight 300 s blocks and four 600 s ones, ending with a full-frame
600 s block bracketed by its own full-frame bias. The exposures are **blocked, not randomised** -
they run short to long - and that is a design choice this session's own finding turns into a
limitation. Section 5 says why.


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10.2, 4.6), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})

b = ped[~ped.full_frame]
ax[0].plot(b.t_min, b.level, "o-", ms=4, lw=0.8, color="0.25", label="bias block level")
ax[0].plot(ped[ped.full_frame].t_min, ped[ped.full_frame].level, "s", ms=5,
           color="tab:purple", label="bias, full frame")
for e, c in zip((1.0, 60.0, 300.0, 600.0), ("0.7", "tab:green", "tab:blue", "crimson")):
    d = roi[roi.exptime == e]
    ax[0].plot(d.t_min, d.level, "o", ms=5, color=c, label=f"dark {e:.0f} s")
ax[0].set(ylabel="plane mean, ADC counts",
          title="the night: every published block against wall clock")
ax[0].legend(ncol=3, fontsize=7)

anom = pd.concat([ped[["t_min", "n_anomalous"]], blocks[["t_min", "n_anomalous"]]])
ax[1].bar(anom.t_min, anom.n_anomalous, width=4, color="crimson")
ax[1].set(xlabel="minutes into the run", ylabel="frames in\nthe far state",
          title="the offset state, per block")
plt.tight_layout()

first = anom[anom.n_anomalous > 0].t_min.min()
print(f"first anomalous frame at {first:.0f} min, in block "
      f"{int(ped.loc[ped.t_min == first, 'block'].iloc[0])}")
print(f"clean before it: {int((anom.t_min < first).sum())} of {len(anom)} blocks, "
      f"{int(anom[anom.t_min < first].n_anomalous.sum())} anomalous frames")
seen = int(anom[anom.t_min >= first].n_anomalous.sum())
total = round(INCIDENCE * N_FRAMES)
print(f"after it: {int((anom.t_min >= first).sum())} blocks holding {seen} anomalous frames")
print(f"\nthe published tables account for {seen} of the {total} the constant counts.  "
      f"The missing {total - seen}\nare block 27's two 600 s darks: both were in the far "
      f"state, so the block has no clean\nframes and no row here at all.  A block can lose "
      f"itself to this.")


## 2. The pedestal series, and what the interleave bought

The bias blocks are a result in their own right. Fifteen pedestal measurements spread over 262
minutes, each a mean of ten frames on a 512-square plane, is the first time this project has
watched the black level across the span an imaging night actually occupies.

**Published: `pedestal_stability` = 0.0206 counts** of residual scatter about a linear trend, with
that trend running at -0.025 counts/hour. Session 01 asked the same question over 15 minutes and
could only bound the rate at +/-0.254 counts/min; this is about 250 times tighter, and it is
tighter because the span is longer, not because the frames are better.

The number matters because of what rule 2 said it would decide. A drift of 0.12 counts across a
bracket would fake a dark current of 1e-4 e-/px/s. The measured wander across a *whole night* is a
sixth of that.


In [ ]:
fit = np.polyfit(b.t_min, b.level, 1)
resid = b.level - np.polyval(fit, b.t_min)

fig, ax = plt.subplots(1, 3, figsize=(10.4, 2.9))
ax[0].plot(b.t_min, b.level, "o", ms=4, color="0.25")
ax[0].plot(b.t_min, np.polyval(fit, b.t_min), "-", lw=0.8, color="crimson",
           label=f"{fit[0] * 60:+.4f} counts/hour")
ax[0].set(xlabel="minutes", ylabel="ADC counts", title="pedestal against wall clock")
ax[0].legend(fontsize=7)

ax[1].axhspan(-K["pedestal_stability"]["value"], K["pedestal_stability"]["value"],
              color="0.88", label="published residual sd")
ax[1].axhline(0, color="0.5", lw=0.6)
ax[1].plot(b.t_min, resid, "o", ms=4, color="0.25")
ax[1].axhline(0.12, color="crimson", lw=0.8, ls="--",
              label="wander that would fake D = 1e-4")
ax[1].axhline(-0.12, color="crimson", lw=0.8, ls="--")
ax[1].set(xlabel="minutes", ylabel="residual, counts", title="residual about the trend")
ax[1].legend(fontsize=7)

for n, c in zip(PLANES, ("tab:red", "tab:olive", "tab:green", "tab:blue")):
    ax[2].plot(b.t_min, b[n] - b[n].iloc[0], "o-", ms=3, lw=0.7, color=c, label=n)
ax[2].set(xlabel="minutes", ylabel="counts from block 0",
          title="all four planes move together")
ax[2].legend(fontsize=7)
plt.tight_layout()

print(f"residual sd re-derived here: {resid.std(ddof=1):.4f} counts "
      f"(published {K['pedestal_stability']['value']})")
print(f"session 01 bounded the drift at +/-0.254 counts/min over 15 min; "
      f"this night measures {abs(fit[0]):.5f} counts/min over {b.t_min.max():.0f} min")


## 3. The finding: the black level has states

The night's identical 600 s dark blocks disagreed by about +/-0.55 counts while the bias series
either side of them was stable to 0.06. Something was moving that the pedestal series could not
see, and three eliminations found it.

**It is a level, not an outlier population.** The scatter is in the *clipped* mean - cosmic rays
and hot pixels live in the tail, and removing the tail did not remove the scatter.

**It is not the interpolation.** Taking the pedestal from the block before, the block after, or
the interpolation between them moves the excess by at most 0.04 counts against a 1.2-count swing.

**It is bimodal.** Per-frame bias levels do not scatter around one value; they sit at **76.48 or
77.51**, and the wild dark block had *both* of its 600 s frames in the far state.

Why this is cheap to detect: read noise on a single pixel is 1.7 counts, but a plane mean over
512x512 pixels averages it down by 512, to about 0.003 counts. The measured within-state scatter
of a plane mean is 0.0072. Against a step of 0.9931 that is a separation of more than a thousand
sigma, so the 0.5-count threshold `07` uses is a formality rather than a judgement call.


In [ ]:
scatter = 0.0072                     # within-state scatter of a plane mean, from 07
sep = STEP / scatter

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.0))

x = np.linspace(-0.4, 1.4, 600)
for mu, c, lab in ((0.0, "0.3", "near state"), (STEP, "crimson", "far state")):
    ax[0].plot(x, np.exp(-0.5 * ((x - mu) / scatter) ** 2), "-", lw=1.0, color=c, label=lab)
ax[0].axvline(0.5, color="tab:blue", lw=0.8, ls="--", label="07's threshold")
ax[0].set(xlabel="plane mean, counts from the near state", yticks=[],
          title=f"the two states, at the scale of a plane mean\n"
                f"separation {sep:.0f} sigma")
ax[0].legend(fontsize=7)

cost = pd.DataFrame({
    "counts": [STEP],
    "electrons": [STEP * G_E],
    "e-/px/s over 600 s": [STEP * G_E / 600],
    "x the 600 s excess": [STEP / abs(roi[roi.exptime == 600].excess.mean())],
    "x the 300 s excess": [STEP / abs(roi[roi.exptime == 300].excess.mean())],
})
ax[1].bar(["read noise\n(1 px)", "state step", "600 s dark\nexcess", "pedestal\nwander"],
          [R_COUNTS, STEP, abs(roi[roi.exptime == 600].excess.mean()),
           K["pedestal_stability"]["value"]],
          color=["0.7", "crimson", "tab:blue", "0.4"])
ax[1].set(ylabel="ADC counts", yscale="log", title="what moves a level, and by how much")
plt.tight_layout()

print(cost.round(4).to_string(index=False))
print(f"\nincidence {100 * INCIDENCE:.1f}% of {N_FRAMES} frames "
      f"= {round(INCIDENCE * N_FRAMES)} frames, none of them in the first 160")
print("uniform across the frame and across all four planes, so it is a single "
      "global level and not a per-plane or per-region effect")


## 4. Why `D` is a bound, and where the bound comes from

A dark current is one number. Read off each exposure separately, this night's is not one number -
it is not even one sign.

| exposure | excess, counts | implied rate, e-/px/s |
|---|---|---|
| 1 s | -0.039 | -2.0e-2 |
| 60 s | +0.118 | +1.0e-3 |
| 300 s | -0.215 | -3.7e-4 |
| 600 s | +0.624 | +5.3e-4 |

**A dark current cannot change sign.** So what is being measured is an offset, and the honest
output is a bound.

**The bound comes from the longest lever arm and nowhere else**, and the first pass of this
analysis got that wrong. Taking the largest implied rate across exposures let the 1 s block set
the "bound": 0.04 counts of offset divided by one second is 2e-2 e-/px/s, a number that says
nothing about dark current and everything about dividing by one second. A fixed level error
divided by an exposure always looks like a rate, and it looks like a *bigger* rate the shorter the
exposure. Only the longest exposure converts a level error into a tight one.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.8, 3.2))

ax[0].axhline(0, color="0.5", lw=0.6)
ax[0].plot(roi.exptime, roi.excess, "o", ms=5, color="0.25")
for _, r in roi.iterrows():
    ax[0].annotate(f"{r.excess:+.2f}", (r.exptime, r.excess), fontsize=6,
                   xytext=(3, 3), textcoords="offset points")
ax[0].axhspan(-STEP / 2, STEP / 2, color="crimson", alpha=0.10,
              label="half the state step")
ax[0].set(xscale="log", xlabel="exposure, s", ylabel="dark minus pedestal, counts",
          title="the excess does not grow with exposure")
ax[0].legend(fontsize=7)

t = np.logspace(0, 3, 100)
ax[1].plot(t, STEP * G_E / t, "-", lw=0.9, color="crimson",
           label="one state step, divided by t")
ax[1].plot(t, K["pedestal_stability"]["value"] * G_E / t, "-", lw=0.9, color="0.5",
           label="pedestal wander, divided by t")
ax[1].axhline(1.3e-6, color="tab:green", lw=0.9, ls="--", label="statistical floor")
ax[1].plot(roi.exptime, roi.implied_e_per_s.abs(), "o", ms=5, color="0.2",
           label="|implied rate| per block")
ax[1].axhline(D_BOUND, color="tab:blue", lw=1.0, label=f"published bound {D_BOUND:.2e}")
ax[1].set(xscale="log", yscale="log", xlabel="exposure, s", ylabel="e-/px/s",
          title="a fixed level error masquerading as a rate")
ax[1].legend(fontsize=6.5)
plt.tight_layout()

stat_floor = 1.3e-6
print(f"what limits the bound: the state, {STEP:.4f} counts "
      f"= {STEP * G_E / 600:.2e} e-/px/s over 600 s")
print(f"what does not:        statistics, at {stat_floor:.1e} e-/px/s "
      f"- a factor of {STEP * G_E / 600 / stat_floor:.0f} below")
print("\nMore frames would not have helped.  That is the protocol's rule 2 confirmed "
      "from the other side:")
print("the error bar was always going to be systematic, and it turned out to be a "
      "systematic nobody had named.")


## 5. What the bound is worth to the model, and the confound it leaves behind

`D` sits inside the shot-noise term, `(F_obj + F_sky + D) * t`. It earns its place only if it is
comparable with `F_sky`. L32's suburban sky rate is 1.594 e-/px/s green; the bound is 5.34e-4.

The cell below prices that. At every exposure this project will use, the dark term is a rounding
error on the sky term - and since the *bound* is an over-estimate of whatever `D` really is, the
real contribution is smaller still. **`D` leaves sigma-squared.**

One nuance the table makes visible: the bound is small against the *sky* term at every exposure,
but it is not small against the *read* term - at 600 s it is 40% of read-noise variance. That is
what a bound looks like when it is set by a systematic rather than by the sensor. It is still an
over-estimate, and the sky term is three orders of magnitude above both, so nothing in the model
turns on it.

**The confound this night cannot resolve.** The 600 s and 300 s dark levels differ by +0.81 counts
against a state step of 0.99. That looks like the two exposures sitting in different states - and
because the ladder was run short to long, *exposure* and *elapsed time* are confounded here in the
way the smoke test's readout-mode question was before it was settled by alternating. If the state turns out to be
exposure-dependent, that is a larger fact than the intermittency, because it would mean every
dark-subtracted long exposure is biased in a way a bias frame cannot see. Resolving it needs
interleaved exposures, and that is a session, not an analysis.


In [ ]:
rows = []
for t in (60, 300, 600, 900):
    sky = SKY_E_PER_S * t
    dark = D_BOUND * t
    read = (R_COUNTS * G_E) ** 2
    rows.append({"t_s": t, "sky var, e-": sky, "dark var, e- (bound)": dark,
                 "read var, e-": read,
                 "dark / sky, %": 100 * dark / sky,
                 "dark / read, %": 100 * dark / read})
print(pd.DataFrame(rows).round(6).to_string(index=False))
print(f"\nthe bound is {SKY_E_PER_S / D_BOUND:,.0f}x below L32's sky rate "
      f"- the dark term contributes {100 * D_BOUND / SKY_E_PER_S:.3f}% of the sky "
      f"variance at any t")

gap = (roi[roi.exptime == 600].level.mean() - roi[roi.exptime == 300].level.mean())
print(f"\nthe open confound: 600 s and 300 s levels differ by {gap:+.3f} counts "
      f"against a state step of {STEP:.3f}")
print(f"as a fraction of a step: {gap / STEP:.2f}")


## 6. `eta_comb` moved to the bias stack, and the stall that sent it there

L15 said per-pixel statistics across a stack are meaningless before registration, and prescribed
frames where pointing does not exist. Darks satisfy that. They then fail for a different reason.

A 32-frame dark stack stalls at **0.930 counts**. Independently, the dark-signal non-uniformity at
300 s measures **0.852 counts** - single-frame spatial variance minus pair-difference variance,
which is `00`'s trick for separating what is fixed from what is random. Those are the same number.
Fixed pattern is identical in every frame of a dark stack and therefore cannot average down, so
the stall is DSNU and not a combination failure.

**This reading was pre-registered.** Protocol rule 4 wrote it down before the data existed, on the
strength of session 02's `fpn_term_present` and session 01's `bias_fixed_pattern_ratio` of 1.011 -
essentially no fixed pattern at bias level. So the constant is measured where the sensor carries
no DSNU: the 143 clean bias frames.

The floor the bias stack approaches is **0.2426 counts**, and session 01's ratio predicts 0.2592
from entirely different frames. Neither session was designed to provide that cross-check.

It remains an **upper bound on the real loss**: no registration, no resampling, and the sky half
unmeasured. What it does buy is a number for the part of the loss that is pure rejection and
averaging, which is the part the model can reason about without a mount.


In [ ]:
N = np.array(sorted(ETA))
eta = np.array([ETA[n] for n in N])

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.1))
ax[0].axhline(1.0, color="0.5", lw=0.8, ls="--", label="ideal sqrt(N)")
ax[0].plot(N, eta, "o-", ms=4, lw=0.9, color="0.25", label="measured, bias stack")
ax[0].set(xscale="log", xlabel="frames combined", ylabel="eta_comb",
          title="combination efficiency against stack size")
ax[0].legend(fontsize=7)

sd1 = 1.0
ax[1].plot(N, sd1 / np.sqrt(N), "--", lw=0.8, color="0.5", label="ideal 1/sqrt(N)")
ax[1].plot(N, sd1 / np.sqrt(N) / eta, "o-", ms=4, lw=0.9, color="0.25", label="measured")
ax[1].set(xscale="log", yscale="log", xlabel="frames combined",
          ylabel="sd of the stack mean, normalised",
          title="where averaging stops paying")
ax[1].legend(fontsize=7)
plt.tight_layout()

print("eta_comb, measured on 143 clean bias frames:")
for n in N:
    bar = "#" * int(round(40 * ETA[n]))
    print(f"  N={n:4d}  {ETA[n]:.4f}  {bar}")
print(f"\nideal to N~8 (eta {ETA[8]:.3f}), then a fixed-pattern floor takes over")
print(f"dark stack stalls at 0.930 counts; DSNU at 300 s is {DSNU:.4f} counts "
      f"= {DSNU * G_E:.4f} e-")
print(f"bias fixed-pattern floor 0.2426 counts; session 01's ratio "
      f"{K1['bias_fixed_pattern_ratio']['value']} predicts 0.2592")


## 7. Glow, and why it survived the state

Protocol rule 5 asked whether `D` is position-dependent, by median-combining the four full-frame
600 s darks against their own full-frame bias. It is: the worst corner sits **+0.333 counts** above
the centre.

**This number is measurable precisely because it is spatial.** The offset state is a single global
level, so it cancels exactly in a difference between two regions of *the same frame*. The pedestal
cancels for the same reason. Two of the night's four results are defined as differences in space
rather than in time, and those two are untouched by the finding that limits the other two - which
is not a coincidence but a design property worth carrying into the next session.

The answer rule 5 wanted: the session ROI reads +0.555 against a centre of +0.541, which is to say
it is in the quiet part of the frame. The model's ROI is not paying for the corners.


In [ ]:
# The per-region means live in the published note, which is a string; parsed rather
# than retyped, so this cell cannot drift away from what results/ says.
note = K["glow_gradient"]["note"]
regions = dict((m, float(v)) for m, v in
               re.findall(r"(centre|top_left|top_right|bot_left|bot_right|session_roi) "
                          r"([+-][0-9]*[.][0-9]+)", note))
order = ["top_left", "top_right", "centre", "bot_left", "bot_right", "session_roi"]

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.1))
colours = ["crimson" if r == "session_roi" else "0.35" for r in order]
ax[0].barh(order, [regions[r] for r in order], color=colours)
ax[0].axvline(regions["centre"], color="tab:blue", lw=0.8, ls="--", label="centre")
ax[0].set(xlabel="counts above bias at 600 s", title="600 s dark, by region")
ax[0].legend(fontsize=7)

grid = np.array([[regions["top_left"], regions["top_right"]],
                 [regions["bot_left"], regions["bot_right"]]])
im = ax[1].imshow(grid, cmap="magma", interpolation="nearest")
ax[1].set(xticks=[], yticks=[], title="the corners, laid out as they sit")
for (i, j), v in np.ndenumerate(grid):
    ax[1].text(j, i, f"{v:+.3f}", ha="center", va="center", color="0.9", fontsize=9)
plt.colorbar(im, ax=ax[1], label="counts")
plt.tight_layout()

worst = max(r for k, r in regions.items() if k not in ("centre", "session_roi"))
print(f"worst corner - centre: {worst - regions['centre']:+.4f} counts "
      f"(published {GLOW:.4f})")
print(f"session ROI - centre:  {regions['session_roi'] - regions['centre']:+.4f} counts")
print(f"as a rate across the frame: {GLOW * G_E / 600:.2e} e-/px/s of spatial variation")
print(f"\nfor scale, the state step is {STEP:.3f} counts - three times the whole "
      f"gradient, and it cancels here exactly")


## 8. What the session settled, and what it did not

**Settled, and available to every later notebook:**

| constant | value | what it unlocks |
|---|---|---|
| `dark_current_bound` | `< 5.34e-4 e-/px/s` at -10 C | `D` leaves sigma-squared; the temperature axis never opens |
| `pedestal_stability` | 0.0206 counts over 262 min | long-exposure dark subtraction is licensed on this rig |
| `eta_comb` | 0.986 / 0.976 / 0.944 at N = 2 / 4 / 8 | the rejection-and-averaging half of MISSION's stacking term |
| `dsnu_300s` | 0.852 counts | why a dark stack stalls, and a fixed-pattern term the model does not have |
| `glow_gradient` | +0.333 counts at 600 s | `D` is position-dependent, and the ROI is in the quiet part |
| `offset_state_step` | 0.9931 counts, 4.1% of frames | a systematic nobody had named |

**And two things that are not constants but are results:**

- **the night held**: 244 of 244 frames in band, 0 retaken, 0 out of band, 5.03 hours. The
  capture half of the protocol needs no revision;
- **rule 1 is refuted as written.** Dark-minus-bias measures the offset state, not the sensor,
  whenever the two frames are not in the same one. The repair is not more frames or tighter
  bracketing; it is classifying the state first.

**Not settled, and worth being explicit about:**

- **why the state exists**, and why it began 161 minutes into a run that changed nothing. This
  night varied nothing that could cause it and therefore cannot answer it;
- **whether the state is exposure-dependent.** Section 5's confound. The exposures ran short to
  long, so a state that depends on exposure and a state that depends on elapsed time look
  identical in this data;
- **whether the step is gain-dependent**, which is the question with teeth: everything in
  `bias_sweep.csv` and `ptc_gain.csv` is a difference against a bias level, so a state that is
  fixed in *electrons* rather than in counts would reach into the read-noise measurement;
- **the sky half of `eta_comb`.** Registration and resampling are unmeasured and will only be
  measurable on real lights.

**The LEGACY queue.** L14 and L15 both have verdicts - one confirmed with its mechanism corrected,
one confirmed and sharpened - and both have left `LEGACY.md`, taking it from 16 entries to 14. The
reasoning is D60 through D64 in `DECISIONS.md`.

**What to do next.** `protocols/04-offset-state.md` is the session this notebook argues for: a
bias-only evening that maps the state against elapsed time, gain, offset and reconfiguration. It
is written to be cheap - a bias frame samples the state at 0.007-count precision and costs nothing
- and it is aimed at the question that threatens later work rather than the one that is merely
interesting. Sharpening `D` is not a goal: the bound already sits three orders of magnitude below
the sky, and a better one would change no recommended exposure and no recommended gain.
